# Prueba de formato: 3 condiciones con Qwen2.5-1.5B

5 preguntas escritas a mano (`data/prueba.jsonl`; 3 chilenas, 2 no chilenas). Objetivo: revisar a ojo el formato de los prompts, el parseo de las acciones de ReAct y las observaciones. **No** mide desempeño. GPU: T4.

In [1]:
!git clone https://github.com/Felipedino/ReAct-Rag-Multimodal.git /content/proyecto

Cloning into '/content/proyecto'...
remote: Enumerating objects: 19107, done.
remote: Total 19107 (delta 0), reused 0 (delta 0), pack-reused 19107 (from 2)
Receiving objects: 100% (19107/19107), 61.10 MiB | 16.05 MiB/s, done.
Resolving deltas: 100% (5290/5290), done.
Updating files: 100% (19574/19574), done.


In [1]:
# Ajustar a donde esté el repo en Colab (clonado o en Drive).
import os
os.chdir('/content/proyecto')
!ls

agents.py  correr.py	  diagnostico_pairs.txt  pairs.py	     specs
cache	   data		  evaluate.py		 prueba_colab.ipynb  wiki.py
CLAUDE.md  DECISIONES.md  llm.py		 questions.py


In [2]:
# Si vllm no está instalado en esta sesión (versión probada: 0.34.0):
!pip install vllm==0.34.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.0/239.0 MB 32.1 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 39.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 MB 36.6 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.0/556.0 kB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 62.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 MB 46.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.0/92.0 MB 45.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 42.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 35.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 21.1 MB/s  0:00:00
   ━━━━━━━━━━━

## 1. Ver los prompts sin cargar el modelo (solo el tokenizer)

In [6]:
from transformers import AutoTokenizer
import json, llm, agents

class SoloTokenizer:
    """Imita a vllm.LLM lo justo para armar prompts sin usar la GPU."""
    def __init__(self, nombre):
        self.tok = AutoTokenizer.from_pretrained(nombre)
    def get_tokenizer(self):
        return self.tok

falso = SoloTokenizer(llm.MODELOS['1.5B'])
q = json.loads(open('data/prueba.jsonl', encoding='utf-8').readline())
usuario = f"""{agents.INSTR_REACT}
Ejemplos:
{agents.EJEMPLOS_REACT}
Pregunta: {q['pregunta']}"""
p = llm.armar_prompt(falso, agents.SISTEMA, usuario, prefijo_asistente='Pensamiento 1:')
print(p)
print('Tokens del prompt ReAct (paso 1):', llm.contar_tokens(falso, p))

<|im_start|>system
Eres un asistente que responde preguntas de conocimiento general en español.<|im_end|>
<|im_start|>user
Responde la pregunta alternando pasos de Pensamiento, Acción y Observación. En el Pensamiento razonas sobre lo que sabes y lo que falta averiguar. La Acción puede ser de dos tipos:
(1) Buscar[entidad]: busca la entidad en Wikipedia y devuelve el comienzo de su artículo. Si no existe, devuelve títulos parecidos para que intentes otra búsqueda.
(2) Terminar[respuesta]: entrega la respuesta corta (un nombre, lugar, número o año) y termina.
Tienes como máximo 7 pasos.
Ejemplos:
Pregunta: ¿En qué localidad nació el compositor que musicalizó el himno nacional de México?
Pensamiento 1: Tengo que buscar el himno nacional de México para saber quién compuso su música, y luego buscar dónde nació esa persona.
Acción 1: Buscar[Himno nacional mexicano]
Observación 1: El Himno nacional mexicano es uno de los tres símbolos patrios establecidos por la ley en dicho país, junto con e

## 2. Correr las 3 condiciones (proceso aparte, libera la GPU al terminar)

In [7]:
!python correr.py 1.5B prueba

W1125 00:04:45.466000 3558 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1125 00:04:45.467000 3558 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1125 00:04:45.467000 3558 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
5 preguntas (prueba); modelo 1.5B; condiciones ['directa', 'rag', 'react']
(MainProcess pid=3558) INFO 11-25 00:04:57 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ'}
(MainProcess pid=3558) WARNING 11-25 00:04:57 [arg_utils.py:2015] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=3558) INFO 11-25 00:05:12 [model.py:761] Resolved architect

## 3. Revisar las trazas

In [8]:
trazas = [json.loads(l) for l in open('resultados/trazas_1.5B_prueba.jsonl', encoding='utf-8')]
for t in trazas:
    print('=' * 100)
    print(f"[{t['condicion']}] {t['pregunta']}")
    print(f"gold: {t['respuesta_gold']} | modelo: {t['respuesta_modelo']!r} | "
          f"termino: {t['termino']} | formato_ok: {t['formato_ok']} | "
          f"acciones inválidas: {t['acciones_invalidas']}")
    if t['titulos_vistos']:
        print('títulos vistos:', t['titulos_vistos'])
    if t['condicion'] == 'react':
        print('-' * 100)
        print(t['texto_modelo'])
    else:
        print('salida cruda:', repr(t['texto_modelo']))

[directa] ¿En qué ciudad nació el compositor español de la música del himno nacional de Chile?
gold: Tárrega | modelo: 'Santiago' | termino: ok | formato_ok: True | acciones inválidas: 0
salida cruda: 'Respuesta: Santiago'
[directa] ¿En qué año murió el poeta chileno que escribió en 1847 la letra del himno nacional de Chile?
gold: 1910 | modelo: '1883' | termino: ok | formato_ok: True | acciones inválidas: 0
salida cruda: 'Respuesta: 1883'
[directa] ¿En qué ciudad nació el autor del poema épico La Araucana?
gold: Madrid | modelo: 'Santiago' | termino: ok | formato_ok: True | acciones inválidas: 0
salida cruda: 'Respuesta: Santiago'
[directa] ¿En qué ciudad nació el arquitecto que diseñó la Ópera de Sídney?
gold: Copenhague | modelo: 'Sydney' | termino: ok | formato_ok: True | acciones inválidas: 0
salida cruda: 'Respuesta: Sydney'
[directa] ¿En qué año murió el director de la película Los siete samuráis?
gold: 1998 | modelo: '1999' | termino: ok | formato_ok: True | acciones inválidas:

Qué mirar:
- directa/rag: `formato_ok` y respuestas cortas (sin explicación).
- react: que cada paso tenga `Acción k: Buscar[...]` o `Terminar[...]`, que el modelo no invente observaciones, que `Buscar` encuentre páginas (si no, ver los 'Similares'), y cuántos pasos usa. Si aparece `termino='contexto'`, subir `llm.MAX_MODEL_LEN`.

In [9]:
!python evaluate.py prueba

EM por celda (filas: modelo, condición; columnas: celda):
grupo             chile_pocos  chile_muchos  no_chile_pocos  no_chile_muchos
modelo condicion                                                            
1.5B   directa            NaN          0.00             NaN              0.0
       rag                NaN          0.00             NaN              0.5
       react              NaN          0.67             NaN              0.5

Ganancia de EM por origen y popularidad:
modelo     comparacion  agrupacion    grupo  n  ganancia_em  ic_lo  ic_hi
  1.5B react - directa      origen    chile  3         0.67    0.0    1.0
  1.5B react - directa      origen no_chile  2         0.50    0.0    1.0
  1.5B react - directa popularidad   muchos  5         0.60    0.2    1.0
  1.5B react - directa       total    total  5         0.60    0.2    1.0
  1.5B     react - rag      origen    chile  3         0.67    0.0    1.0
  1.5B     react - rag      origen no_chile  2         0.00    0.0    0

Traer cambios

In [10]:
!git pull

remote: Enumerating objects: 1293, done.
remote: Counting objects: 100% (1293/1293), done.
remote: Compressing objects: 100% (805/805), done.
remote: Total 1293 (delta 487), reused 1293 (delta 487), pack-reused 0 (from 0)
Receiving objects: 100% (1293/1293), 5.59 MiB | 4.52 MiB/s, done.
Resolving deltas: 100% (487/487), completed with 9 local objects.
From https://github.com/Felipedino/ReAct-Rag-Multimodal
   a382f664..7d192d89  main       -> origin/main
Updating a382f664..7d192d89
Fast-forward
 DECISIONES.md                                      |  55 ++
 .../0001fb0b6010d37c05ea639576524691b53d43be.json  |   1 +
 .../002b2c3a071bc2cf1a719df2140d88744522b72e.json  |   1 +
 .../004a0f54be2748bc22a0a26c55e7c433d30cfd92.json  |   1 +
 .../0051c19fa5a011dc48ba388ed27fe46d098cf27f.json  |   1 +
 .../006349803cbb2196efbdc5bf6b81466a7de0723c.json  |   1 +
 .../00ec8e601de8c8125577d5a4f2ce220b16c6f056.json  |   1 +
 .../014ae7388772011e917d7b49d0f6e3453bdaf97f.json  |   1 +
 .../01646417335c81

Corrida dev para revisar q funcione

In [11]:
!python correr.py 1.5B dev
!python correr.py 7B dev
!python evaluate.py dev

W1125 00:24:51.250000 8905 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1125 00:24:51.251000 8905 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1125 00:24:51.251000 8905 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
20 preguntas (dev); modelo 1.5B; condiciones ['directa', 'rag', 'react']
(MainProcess pid=8905) INFO 11-25 00:25:03 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ'}
(MainProcess pid=8905) WARNING 11-25 00:25:03 [arg_utils.py:2015] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=8905) INFO 11-25 00:25:03 [model.py:761] Resolved architectur

In [12]:
import json
tr = [json.loads(l) for l in open('resultados/trazas_1.5B_dev.jsonl', encoding='utf-8')]
mp = [t for t in tr if t['condicion'] == 'react' and t['termino'] == 'max_pasos']
for t in mp[:3]:
    print('=' * 80); print(t['pregunta'], '| gold:', t['respuesta_gold'])
    for i, p in enumerate(t['pasos'], 1):
        print(f"{i}. {p['accion']}[{p['argumento']}] -> {p['titulo_observado']}")
        print('   pensamiento:', (p['pensamiento'] or '')[:200])

¿En qué año se inauguró la estación de la línea 5 del Metro ubicada bajo la plaza en cuya vereda centro-norte se encuentra el Palacio de la Real Audiencia? | gold: 2000
1. Buscar[Estación de la línea 5 del Metro] -> None
   pensamiento: Tengo que buscar la estación de la línea 5 del Metro para saber en qué año se inauguró, y luego buscar dónde está ubicada la plaza en cuya vereda centro-norte se encuentra el Palacio de la Real Audien
2. Buscar[Estación de la línea 5 del Metro de Barcelona] -> None
   pensamiento: No se encontró la estación de la línea 5 del Metro, por lo que tengo que buscar la estación de la línea 5 del Metro de Barcelona para poder encontrar la información.
3. Buscar[Estación de la línea 5 del Metro de Madrid] -> None
   pensamiento: No se encontró la estación de la línea 5 del Metro de Barcelona, por lo que tengo que buscar la estación de la línea 5 del Metro de Madrid para poder encontrar la información.
4. Buscar[Estación de la línea 5 del Metro de Barcelona] -> N

corrida real test

In [13]:
!python correr.py 1.5B test
!python correr.py 3B test
!python correr.py 7B test
!python evaluate.py test

W1125 00:49:17.300000 15358 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1125 00:49:17.300000 15358 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1125 00:49:17.301000 15358 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
160 preguntas (test); modelo 1.5B; condiciones ['directa', 'rag', 'react']
(MainProcess pid=15358) INFO 11-25 00:49:31 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ'}
(MainProcess pid=15358) WARNING 11-25 00:49:32 [arg_utils.py:2015] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=15358) INFO 11-25 00:49:32 [model.py:761] Resolved arc

guardar resultados

In [14]:
!zip -qr salida.zip resultados cache
from google.colab import files; files.download("salida.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
%cd /content/proyecto
!mv resultados resultados_colab_viejo
!git pull


/content
Updating 7d192d89..cc95a42f
Fast-forward
 DECISIONES.md                       |  48 +++
 evaluate.py                         |   2 +-
 llm.py                              |   7 +-
 resultados/analisis_errores.md      | 123 ++++++
 resultados/em_dev.png               | Bin 0 -> 27571 bytes
 resultados/em_prueba.png            | Bin 0 -> 23841 bytes
 resultados/em_test.png              | Bin 0 -> 31424 bytes
 resultados/errores_dev.csv          |  64 +++
 resultados/errores_prueba.csv       |   7 +
 resultados/errores_test.csv         | 749 ++++++++++++++++++++++++++++++++++++
 resultados/figura_principal.png     | Bin 0 -> 121091 bytes
 resultados/ganancia_dev.csv         |  55 +++
 resultados/ganancia_prueba.csv      |  19 +
 resultados/ganancia_test.csv        |  82 ++++
 resultados/tabla_dev.csv            |  55 +++
 resultados/tabla_prueba.csv         |  19 +
 resultados/tabla_test.csv           |  82 ++++
 resultados/trazas_1.5B_dev.jsonl    |  60 +++
 resultados/trazas_1.

In [18]:
!sed -i 's/^MAX_MODEL_LEN = 8192/MAX_MODEL_LEN = 6144/' llm.py

!python correr.py 14B test directa react

W1125 02:13:00.032000 36954 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1125 02:13:00.032000 36954 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1125 02:13:00.033000 36954 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
160 preguntas (test); modelo 14B; condiciones ['directa', 'react']
(MainProcess pid=36954) INFO 11-25 02:13:18 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 6144, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-14B-Instruct-AWQ'}
(MainProcess pid=36954) WARNING 11-25 02:13:18 [arg_utils.py:2015] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=36954) INFO 11-25 02:13:18 [model.py:761] Resolved architecture